# 第 03 章：RAG 2.0 增强型数据检索

> **学习定位**：本章属于「第一阶段：把 LangChain 当开发入口」。
> 这一章的 notebook 目标很明确：把外部知识稳稳接到模型前面，亲手跑通一条从资料入库到最终回答的最小 RAG 链。

本章实验会按这个顺序推进：
1. 构造一份最小知识库
2. 载入文档并切分
3. 建立向量索引与增量管理
4. 对比 BM25、向量检索、混合检索
5. 把检索结果接回 `prompt | llm | parser`


## 1. 准备一份最小知识库

先不要引入复杂数据源，我们就在当前目录下构造一份极简 `knowledge.txt`。

In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

knowledge_path = Path('knowledge.txt')
knowledge_path.write_text(
    '\n'.join([
        '1001-AABBCC: 这是一张购买于 2026 年的主板，核心组件为 Z170X，保修期五年。',
        '今天公司发布了新规，要求员工早上 9:30 之前全部打卡完毕，迟到扣减工资 100 元。',
        '若遇到系统卡顿问题，请务必尝试长按电源键 5 秒强制重启，这是硬件级的保险措施。',
        '客服值班表更新：周一至周五 09:00-18:00 在线，周末只处理 P0 级紧急问题。',
    ]),
    encoding='utf-8',
)

print('知识文件已生成：')
print(knowledge_path.read_text(encoding='utf-8'))


知识文件已生成：
1001-AABBCC: 这是一张购买于 2026 年的主板，核心组件为 Z170X，保修期五年。
今天公司发布了新规，要求员工早上 9:30 之前全部打卡完毕，迟到扣减工资 100 元。
若遇到系统卡顿问题，请务必尝试长按电源键 5 秒强制重启，这是硬件级的保险措施。
客服值班表更新：周一至周五 09:00-18:00 在线，周末只处理 P0 级紧急问题。


## 2. 载入并切分文档

这一节先建立最小预处理链路：`Loader -> Splitter`。

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader(str(knowledge_path), encoding='utf-8')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
)
splits = text_splitter.split_documents(docs)

print(f'原始文档数: {len(docs)}')
print(f'切分后文档块数: {len(splits)}')
print('第一块 metadata:', splits[0].metadata)
print('第一块内容:', splits[0].page_content)


原始文档数: 1
切分后文档块数: 2
第一块 metadata: {'source': 'knowledge.txt'}
第一块内容: 1001-AABBCC: 这是一张购买于 2026 年的主板，核心组件为 Z170X，保修期五年。
今天公司发布了新规，要求员工早上 9:30 之前全部打卡完毕，迟到扣减工资 100 元。


## 3. 初始化 Embeddings

这里继续沿用课程里的统一设置，把文本块映射到向量空间。

In [3]:
from langchain.embeddings import init_embeddings

embeddings = init_embeddings(
    model='text-embedding-v4',
    provider='openai',
    api_key=os.getenv('BAILIAN_API_KEY'),
    base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
    check_embedding_ctx_length=False,
)

print('Embedding 模型初始化完成。')


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

## 4. 建立向量库与增量索引

这一格可以重复执行。第一次通常会看到 `added`，后续重复运行时应该更多看到 `skipped`，这就是 `SQLRecordManager` 在帮我们做增量治理。

In [ ]:
from langchain_chroma import Chroma
from langchain_classic.indexes import SQLRecordManager, index

vectorstore = Chroma(
    collection_name='my_private_knowledge',
    embedding_function=embeddings,
    persist_directory='./chroma_db',
)

record_manager = SQLRecordManager(
    'chroma/my_private_knowledge',
    db_url='sqlite:///record_manager_cache.sql',
)
record_manager.create_schema()

index_result = index(
    docs_source=splits,
    record_manager=record_manager,
    vector_store=vectorstore,
    cleanup='incremental',
    source_id_key='source',
)

print('本次入库结果:', index_result)


## 5. 先观察不同检索器的召回风格

RAG 的关键不是只会建库，而是能看懂不同检索器到底各自擅长什么。

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

query = '我的编号是 1001-AABBCC，请问它是什么设备？'

bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 2

vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

def show_docs(title, docs):
    print(f'\n--- {title} ---')
    if not docs:
        print('没有召回到结果')
        return
    for idx, doc in enumerate(docs, start=1):
        print(f'[{idx}] {doc.page_content}')

bm25_docs = bm25_retriever.invoke(query)
vector_docs = vector_retriever.invoke(query)
hybrid_docs = hybrid_retriever.invoke(query)

print('问题:', query)
show_docs('BM25 召回', bm25_docs)
show_docs('向量检索召回', vector_docs)
show_docs('混合检索召回', hybrid_docs)


## 6. 给中文 BM25 补上分词

如果你发现编号、专有名词、中文短语命中得不稳定，可以显式给 BM25 加分词预处理。

In [ ]:
import jieba
from langchain_community.retrievers import BM25Retriever

def chinese_tokenizer(text: str):
    return list(jieba.cut(text))

bm25_retriever = BM25Retriever.from_documents(
    splits,
    preprocess_func=chinese_tokenizer,
)
bm25_retriever.k = 2

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)

bm25_docs = bm25_retriever.invoke(query)
hybrid_docs = hybrid_retriever.invoke(query)

show_docs('补分词后的 BM25 召回', bm25_docs)
show_docs('补分词后的混合检索召回', hybrid_docs)


## 7. 把检索接回一条普通的 LLM 链

这一步最重要：RAG 的最后一段，仍然是一条普通的 Runnable 链。

In [ ]:
from operator import itemgetter

from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model(
    model='deepseek-chat',
    model_provider='deepseek',
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com',
)

system_prompt = '''你是一个专业的知识库助手。
请根据提供的【线索】来回答用户的【问题】。
如果线索中没有答案，请明确说明不知道，不要编造。'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '【问题】：{question}\n\n【线索】：{context}'),
])

def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {
        'question': itemgetter('question'),
        'context': itemgetter('question') | hybrid_retriever | format_docs,
    }
    | prompt
    | llm
    | StrOutputParser()
)

print('RAG 链已准备完成。')


## 8. 运行最终 RAG 闭环

这里我们分别问一个编号问题和一个制度问题，确认同一条链都能工作。

In [ ]:
question_1 = '我的编号是 1001-AABBCC，请问它是什么设备？'
answer_1 = rag_chain.invoke({'question': question_1})
print('问题 1:', question_1)
print('回答 1:', answer_1)

question_2 = '公司现在要求员工几点之前完成打卡？'
answer_2 = rag_chain.invoke({'question': question_2})
print('\n问题 2:', question_2)
print('回答 2:', answer_2)


## 9. 本章小结

如果这一章跑通了，你应该已经建立起一个很关键的工程直觉：

- RAG 不是 Agent 的同义词
- RAG 的核心是一条可以拆开的检索增强链路
- 检索结束后，最终仍然要回到 `prompt | llm | parser`
